# Atmospheric seeing options: delivered FWHM vs DIMM and RINGSS as a function of wind

This notebook compares a few ways of predicting the atmosphere seeing, to try to decide which is the best method. We use
LSSTCam science visits from the Consolidated Database
(`cdb_lsstcam.visit1` + `cdb_lsstcam.visit1_quicklook`, schema: https://sdm-schemas.lsst.io/cdb_lsstcam.html)
and RINGSS measurements from the EFD (`lsst.sal.ESS.logevent_ringssMeasurement`, `salIndex = 304`).

For every visit we form

* **FWHM(zenith, 500 nm)** = `psf_sigma_median` × 2.355 × 0.2″/pix × airmass$^{-0.6}$ × $(\lambda_\mathrm{eff}/500\,\mathrm{nm})^{0.2}$
  (the same airmass and bandpass corrections as `lsst.summit.utils`; the stored
  `seeing_zenith_500nm_median` column has been empty for all of early operations, so it is recomputed here).
* **DIMM, outer-scale corrected** (von Kármán, Tokovinin 2002) for each outer scale $L_0$ option:

$$ r_0 = \frac{0.98 \cdot 500\,\mathrm{nm}}{\theta_\mathrm{DIMM}\,[\mathrm{rad}]}, \qquad
   \theta_{L_0} = \theta_\mathrm{DIMM}\sqrt{1 - 2.18\,(r_0/L_0)^{0.356}} $$

  with the default options *none* (Kolmogorov, $L_0 = \infty$), 25 m and 30 m.
* **RINGSS** total seeing (`fwhmScintillation`, from the turbulence profile) and the alternative
  `fwhmSector` estimate, each taken as the median of the RINGSS measurements within a few minutes of
  the visit midpoint. RINGSS reports seeing at zenith and 500 nm, so no further correction is applied
  (switchable below).
* **Δ = FWHM(zenith, 500 nm) − reference seeing**, plotted against outside wind speed as hexbin density
  and binned medians (16–84 % band, robust error on the median), for every reference.

Δ > 0 is blur the monitor does not see (dome/mirror seeing, wind shake, optics); Δ < 0 means the monitor
over-predicts the delivered seeing.

**Parameters** (first code cell): date range, science programs (defaults BLOCK-407, 408, 416, 417, 419, 421),
outer-scale options, RINGSS matching window, and the data source (`consdb` on USDF, or `local` for the
CSV scrapes in `../consdb_tables` and `../efd_tables`). Everything else is defined in this notebook.


In [ ]:
# Parameters cell. Times Square substitutes these as strings; they are coerced below.
day_obs_min = 20251025  # first night (YYYYMMDD)
day_obs_max = 20260714  # last night (YYYYMMDD)
science_programs = "407, 408, 416, 417, 419, 421"  # bare numbers become BLOCK-<n>
outer_scales = (
    "none, 25, 30"  # DIMM outer-scale options in metres ("none" = Kolmogorov)
)
ringss_window_minutes = (
    5  # RINGSS measurements within +/- this of the visit midpoint are medianed
)
ringss_at_zenith = (
    True  # RINGSS seeing is already zenith/500 nm; set False to apply cos(zd)^0.6
)
data_source = (
    "consdb"  # "consdb" (USDF / RSP: ConsDB + EFD) or "local" (CSV scrapes in ../)
)
efd_name = (
    "usdf_efd"  # lsst_efd_client instance name (ignored for data_source = "local")
)
use_cache = True  # cache the ConsDB/EFD pulls as parquet under ./cache (ignored on Times Square)

not_times_square = True

In [ ]:
# Times Square runs this notebook in a Nublado pod where rubin_nights may be
# missing or stale; install/upgrade it there. Anywhere else this cell is a no-op.
try:
    not_times_square
except NameError:
    not_times_square = False

if not not_times_square:
    print("updating rubin_nights")
    !pip install --user --upgrade git+https://github.com/lsst-sims/rubin_nights.git --no-deps > /dev/null 2>&1

In [ ]:
import glob
import inspect
import os
import re
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Iterable, Sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from astropy.time import Time

%matplotlib inline
pd.set_option("display.width", 160)
NOTEBOOK_DIR = os.getcwd()

## Constants and parameter helpers

In [ ]:
DEFAULT_PROGRAMS = (
    "BLOCK-407",
    "BLOCK-408",
    "BLOCK-416",
    "BLOCK-417",
    "BLOCK-419",
    "BLOCK-421",
)
DEFAULT_OUTER_SCALES = (None, 25.0, 30.0)  # metres; None = no outer-scale correction

SIGMA_TO_FWHM = 2.0 * np.sqrt(2.0 * np.log(2.0))
PLATESCALE = 0.2  # arcsec / pixel, LSSTCam
ARCSEC_TO_RAD = np.pi / (180.0 * 3600.0)
REFERENCE_WAVELENGTH_NM = 500.0

# Effective wavelengths used by lsst.summit.utils.getBandpassSeeingCorrection for LSSTCam.
LSSTCAM_EFFECTIVE_WAVELENGTH_NM = {
    "u": 365.1397,
    "g": 475.6104,
    "r": 618.6055,
    "i": 753.5490,
    "z": 868.0935,
    "y": 972.5979,
}
BAND_ORDER = ("u", "g", "r", "i", "z", "y")

# Columns pulled from cdb_lsstcam.visit1 (per-visit metadata) and visit1_quicklook (image quality).
VISIT_COLUMNS = (
    "visit_id",
    "day_obs",
    "seq_num",
    "band",
    "physical_filter",
    "science_program",
    "img_type",
    "target_name",
    "exp_time",
    "exp_midpt",
    "exp_midpt_mjd",
    "airmass",
    "altitude",
    "azimuth",
    "zenith_distance",
    "air_temp",
    "pressure",
    "humidity",
    "wind_speed",
    "wind_dir",
    "dimm_seeing",
)
QUICKLOOK_COLUMNS = (
    "psf_sigma_median",
    "psf_sigma_min",
    "psf_sigma_max",
    "seeing_zenith_500nm_median",
    "n_psf_star_total",
    "psf_ixx_median",
    "psf_iyy_median",
    "psf_ixy_median",
    "aos_fwhm",
    "donut_blur_fwhm",
    "ringss_seeing",
    "guider_total_seeing",
    "guider_ground_layer_seeing",
    "guider_free_seeing",
    "guider_psf_fwhm",
)

# RINGSS event in the EFD.
RINGSS_TOPIC = "lsst.sal.ESS.logevent_ringssMeasurement"
RINGSS_SAL_INDEX = 304
RINGSS_FIELDS = (
    "timestamp",
    "hrNum",
    "zenithDistance",
    "flux",
    "fwhmScintillation",
    "fwhmSector",
    "fwhmFree",
    "wind",
    "tau0",
    "theta0",
)

# Chart palette: fixed categorical order (blue, orange, aqua, yellow, magenta, green, violet, red).
SERIES_COLORS = (
    "#2a78d6",
    "#eb6834",
    "#1baf7a",
    "#eda100",
    "#e87ba4",
    "#008300",
    "#4a3aa7",
    "#e34948",
)
SERIES_MARKERS = ("o", "s", "^", "D", "v", "P", "X", "*")
INK, INK_SECONDARY, INK_MUTED, GRID, SURFACE = (
    "#0b0b0b",
    "#52514e",
    "#898781",
    "#e1e0d9",
    "#fcfcfb",
)
SEQUENTIAL_BLUE = LinearSegmentedColormap.from_list(
    "seq_blue",
    [
        "#ffffff",
        "#cde2fb",
        "#9ec5f4",
        "#6da7ec",
        "#3987e5",
        "#256abf",
        "#1c5cab",
        "#104281",
        "#0d366b",
    ],
)

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#c3c2b7",
        "axes.labelcolor": INK,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "axes.axisbelow": True,
        "xtick.color": INK_SECONDARY,
        "ytick.color": INK_SECONDARY,
        "legend.frameon": False,
        "font.size": 11,
        "axes.titlesize": 12,
        "lines.linewidth": 2.0,
        "savefig.dpi": 150,
    }
)


def parse_day_obs(value) -> int:
    """Coerce 20260101, "20260101" or "2026-01-01" to an int."""
    s = str(value).strip().replace("-", "")
    if not re.fullmatch(r"\d{8}", s):
        raise ValueError(f"Not a day_obs: {value!r}")
    return int(s)


def parse_programs(value) -> list:
    """List or comma/space separated string of block names; bare numbers become BLOCK-<n>."""
    if value is None:
        return list(DEFAULT_PROGRAMS)
    items = (
        [v for v in re.split(r"[,\s]+", value.strip()) if v]
        if isinstance(value, str)
        else [str(v).strip() for v in value]
    )
    return [f"BLOCK-{it}" if re.fullmatch(r"T?\d+", it) else it for it in items]


def parse_outer_scales(value) -> list:
    """Coerce "none,25,30" or [None, 25, 30] to a list of floats/None."""
    if value is None:
        return list(DEFAULT_OUTER_SCALES)
    items = (
        [v for v in re.split(r"[,\s]+", value.strip()) if v]
        if isinstance(value, str)
        else list(value)
    )
    return [
        (
            None
            if (it is None or str(it).strip().lower() in ("none", "inf", "nan", ""))
            else float(it)
        )
        for it in items
    ]


def outer_scale_tag(L0) -> str:
    """Reference tag for an outer-scale option: dimm_none, dimm_L25, ..."""
    if L0 is None:
        return "dimm_none"
    return (
        f"dimm_L{int(L0)}"
        if float(L0).is_integer()
        else "dimm_L" + str(L0).replace(".", "p")
    )


def outer_scale_label(L0) -> str:
    return "DIMM, no L0 correction" if L0 is None else f"DIMM, L0 = {L0:g} m"


def day_obs_chunks(day_obs_min: int, day_obs_max: int, chunk_days: int):
    """Yield (lo, hi) day_obs pairs covering the range in calendar chunks."""
    lo, end = pd.Timestamp(str(day_obs_min)), pd.Timestamp(str(day_obs_max))
    while lo <= end:
        hi = min(lo + pd.Timedelta(days=chunk_days - 1), end)
        yield int(lo.strftime("%Y%m%d")), int(hi.strftime("%Y%m%d"))
        lo = hi + pd.Timedelta(days=1)


def day_obs_window(day_obs_lo: int, day_obs_hi: int):
    """UTC start/end (astropy Time) covering the nights day_obs_lo..day_obs_hi: 12:00 UTC to 12:00 UTC next day."""
    t0 = datetime.strptime(str(day_obs_lo), "%Y%m%d").replace(
        hour=12, tzinfo=timezone.utc
    )
    t1 = datetime.strptime(str(day_obs_hi), "%Y%m%d").replace(
        hour=12, tzinfo=timezone.utc
    ) + pd.Timedelta(days=1)
    return Time(t0), Time(t1)


def _read_cache(cache_path, use_cache):
    if cache_path and use_cache and os.path.exists(cache_path):
        df = pd.read_parquet(cache_path)
        print(f"Loaded {len(df)} rows from cache {cache_path}")
        return df
    return None


def _write_cache(cache_path, df):
    if cache_path and len(df):
        os.makedirs(os.path.dirname(os.path.abspath(cache_path)), exist_ok=True)
        df.to_parquet(cache_path, index=False)
        print(f"Saved {len(df)} rows to {cache_path}")
    return df


def cached(cache_path, loader, use_cache=True) -> pd.DataFrame:
    """Run loader() unless cache_path already holds its parquet output."""
    df = _read_cache(cache_path, use_cache)
    return df if df is not None else _write_cache(cache_path, loader())


async def cached_async(cache_path, loader, use_cache=True) -> pd.DataFrame:
    """Same as cached() for a loader that returns a coroutine (the EFD client is async on the RSP)."""
    df = _read_cache(cache_path, use_cache)
    return df if df is not None else _write_cache(cache_path, await loader())

## ConsDB access

One row per science visit. On USDF the client comes from `rubin_nights` (falls back to
`lsst.summit.utils.ConsDbClient`); the query is chunked by calendar month to stay under the API timeout.


In [ ]:
USDF_CONSDB_URL = "http://consdb-pq.consdb:8080/consdb"


class _SummitUtilsClient:
    """Adapter so lsst.summit.utils.ConsDbClient.query returns a DataFrame."""

    def __init__(self, url=USDF_CONSDB_URL):
        from lsst.summit.utils import ConsDbClient

        os.environ["no_proxy"] = os.environ.get("no_proxy", "") + ",.consdb"
        self._client = ConsDbClient(url)

    def query(self, sql):
        result = self._client.query(sql)
        return result if isinstance(result, pd.DataFrame) else result.to_pandas()


def get_consdb_client():
    """Object with .query(sql) -> DataFrame: rubin_nights first, then summit_utils."""
    errors = []
    try:
        from rubin_nights import connections

        client = connections.get_clients()["consdb"]
        print("ConsDB client: rubin_nights")
        return client
    except Exception as exc:
        errors.append(f"rubin_nights: {exc!r}")
    try:
        client = _SummitUtilsClient()
        print("ConsDB client: lsst.summit.utils")
        return client
    except Exception as exc:
        errors.append(f"summit_utils: {exc!r}")
    raise RuntimeError("Could not build a ConsDB client:\n  " + "\n  ".join(errors))


def _sql_list(values):
    return ", ".join("'" + str(v).replace("'", "''") + "'" for v in values)


def build_query(
    day_obs_min,
    day_obs_max,
    programs=DEFAULT_PROGRAMS,
    instrument="lsstcam",
    img_types=("science",),
):
    """SQL joining visit1 (pointing, weather, DIMM) to visit1_quicklook (PSF stats), one row per visit."""
    schema = f"cdb_{instrument}"
    where = [f"v.day_obs >= {int(day_obs_min)}", f"v.day_obs <= {int(day_obs_max)}"]
    if programs:
        where.append(f"v.science_program IN ({_sql_list(programs)})")
    if img_types:
        where.append(f"v.img_type IN ({_sql_list(img_types)})")
    return (
        "SELECT\n    "
        + ",\n    ".join(
            [f"v.{c}" for c in VISIT_COLUMNS] + [f"q.{c}" for c in QUICKLOOK_COLUMNS]
        )
        + f"\nFROM {schema}.visit1 AS v"
        + f"\nLEFT JOIN {schema}.visit1_quicklook AS q ON q.visit_id = v.visit_id"
        + "\nWHERE "
        + "\n  AND ".join(where)
        + "\nORDER BY v.day_obs, v.seq_num"
    )


def fetch_visits(
    client, day_obs_min, day_obs_max, programs=DEFAULT_PROGRAMS, chunk_days=31
):
    frames = []
    for lo, hi in day_obs_chunks(day_obs_min, day_obs_max, chunk_days):
        df = client.query(build_query(lo, hi, programs))
        print(f"  consdb {lo}-{hi}: {len(df):6d} visits")
        if len(df):
            frames.append(df)
    if not frames:
        return pd.DataFrame(columns=list(VISIT_COLUMNS) + list(QUICKLOOK_COLUMNS))
    return (
        pd.concat(frames, ignore_index=True)
        .sort_values(["day_obs", "seq_num"])
        .reset_index(drop=True)
    )

## EFD access: RINGSS

`fetch_ringss` pulls every `logevent_ringssMeasurement` (salIndex 304) over the night range with
`lsst_efd_client` (as in the AOS nightly report notebook). `match_ringss_to_visits` then attaches, to each
visit, the median of the measurements within ± `ringss_window_minutes` of the exposure midpoint
(`ringss_scint`, `ringss_sector`, `ringss_free`, `ringss_wind`, `ringss_zd`) and the number used (`ringss_n`).

In [ ]:
def get_efd_client(efd_name="usdf_efd"):
    import lsst_efd_client

    return lsst_efd_client.EfdClient(efd_name, output_mode="dataframe")


async def fetch_ringss(
    efd_client, day_obs_min, day_obs_max, chunk_days=31, sal_index=RINGSS_SAL_INDEX
):
    """All RINGSS measurements between 12:00 UTC on day_obs_min and 12:00 UTC the day after day_obs_max."""
    frames = []
    for lo, hi in day_obs_chunks(day_obs_min, day_obs_max, chunk_days):
        start, end = day_obs_window(lo, hi)
        df = efd_client.select_time_series(
            RINGSS_TOPIC, list(RINGSS_FIELDS), start, end, index=sal_index
        )
        if inspect.isawaitable(df):
            df = await df
        print(f"  ringss {lo}-{hi}: {len(df):6d} measurements")
        if len(df):
            df = df.reset_index().rename(columns={df.index.name or "index": "time"})
            frames.append(df)
    if not frames:
        return pd.DataFrame(columns=["time", *RINGSS_FIELDS])
    out = pd.concat(frames, ignore_index=True)
    out["time"] = pd.to_datetime(out["time"], utc=True).dt.tz_localize(None)
    return out.sort_values("time").reset_index(drop=True)


def match_ringss_to_visits(visits, ringss, window_minutes=5.0, at_zenith=True):
    """Median RINGSS quantities within +/- window of each visit midpoint.

    Returns a copy of ``visits`` with ringss_scint, ringss_sector, ringss_free, ringss_wind,
    ringss_zd, ringss_n. With at_zenith=False the two seeing estimates are multiplied by
    cos(zenithDistance)^0.6 to refer them to zenith.
    """
    out = visits.copy()
    cols = {
        "ringss_scint": "fwhmScintillation",
        "ringss_sector": "fwhmSector",
        "ringss_free": "fwhmFree",
        "ringss_wind": "wind",
        "ringss_zd": "zenithDistance",
    }
    for c in cols:
        out[c] = np.nan
    out["ringss_n"] = 0
    if ringss is None or len(ringss) == 0:
        return out
    r = ringss.dropna(subset=["time"]).sort_values("time")
    t_r = r["time"].to_numpy(dtype="datetime64[ns]").astype("int64") / 1e9
    vals = {
        c: r[src].to_numpy(dtype=float) if src in r else np.full(len(r), np.nan)
        for c, src in cols.items()
    }
    if not at_zenith and "zenithDistance" in r:
        corr = np.cos(np.deg2rad(vals["ringss_zd"])) ** 0.6
        vals["ringss_scint"] = vals["ringss_scint"] * corr
        vals["ringss_sector"] = vals["ringss_sector"] * corr
        vals["ringss_free"] = vals["ringss_free"] * corr
    t_v = (
        pd.to_datetime(out["exp_midpt"], utc=True, format="ISO8601")
        .dt.tz_localize(None)
        .to_numpy(dtype="datetime64[ns]")
        .astype("int64")
        / 1e9
    )
    w = window_minutes * 60.0
    lo = np.searchsorted(t_r, t_v - w, side="left")
    hi = np.searchsorted(t_r, t_v + w, side="right")
    res = {c: np.full(len(out), np.nan) for c in cols}
    n = np.zeros(len(out), dtype=int)
    for i, (a, b) in enumerate(zip(lo, hi)):
        if b > a:
            n[i] = b - a
            for c in cols:
                seg = vals[c][a:b]
                seg = seg[np.isfinite(seg)]
                if len(seg):
                    res[c][i] = np.median(seg)
    for c in cols:
        out[c] = res[c]
    out["ringss_n"] = n
    return out

## Physics: zenith/500 nm correction, outer-scale DIMM correction, references

A *reference* is any monitor-based seeing estimate we compare the delivered FWHM against. Each has a
tag, a label and a column: `dimm_none`, `dimm_L25`, `dimm_L30`, ... for the DIMM options and
`ringss_scint`, `ringss_sector` for RINGSS. `add_derived_columns` adds, per reference,
`delta_<tag>` = FWHM(zenith, 500 nm) − reference and `excess_quad_<tag>` = $\sqrt{\mathrm{FWHM}^2 - \mathrm{ref}^2}$.

In [ ]:
def airmass_seeing_correction(airmass):
    """Multiply an observed FWHM by this to refer it to zenith (X^-0.6)."""
    airmass = np.asarray(airmass, dtype=float)
    return np.where(airmass >= 1.0, airmass ** (-0.6), np.nan)


def bandpass_seeing_correction(band):
    """Multiply an observed FWHM by this to refer it to 500 nm; accepts 'r' or 'r_57'."""
    first = pd.Series(band).astype("string").str.strip().str[0]
    wl = first.map(LSSTCAM_EFFECTIVE_WAVELENGTH_NM).astype(float)
    return (wl.to_numpy() / REFERENCE_WAVELENGTH_NM) ** 0.2


def fwhm_zenith_500nm(psf_sigma_pix, airmass, band):
    fwhm = np.asarray(psf_sigma_pix, dtype=float) * SIGMA_TO_FWHM * PLATESCALE
    return fwhm * airmass_seeing_correction(airmass) * bandpass_seeing_correction(band)


def fried_parameter(seeing_arcsec, wavelength_nm=REFERENCE_WAVELENGTH_NM):
    """r0 (m) from a Kolmogorov FWHM in arcsec: r0 = 0.98 lambda / FWHM."""
    with np.errstate(divide="ignore", invalid="ignore"):
        return (
            0.98
            * wavelength_nm
            * 1e-9
            / (np.asarray(seeing_arcsec, dtype=float) * ARCSEC_TO_RAD)
        )


def outer_scale_correction(seeing_arcsec, L0):
    """sqrt(1 - 2.18 (r0/L0)^0.356) (Tokovinin 2002); 1 for L0=None; NaN where the bracket is negative."""
    seeing = np.asarray(seeing_arcsec, dtype=float)
    if L0 is None:
        return np.where(np.isfinite(seeing), 1.0, np.nan)
    with np.errstate(invalid="ignore"):
        bracket = 1.0 - 2.18 * np.power(fried_parameter(seeing) / float(L0), 0.356)
        return np.where(bracket > 0, np.sqrt(np.clip(bracket, 0, None)), np.nan)


def make_references(L0_options, include_ringss=True):
    """[(tag, label), ...] in plotting order: DIMM options first, then RINGSS."""
    refs = [(outer_scale_tag(L0), outer_scale_label(L0)) for L0 in L0_options]
    if include_ringss:
        refs += [("ringss_scint", "RINGSS total"), ("ringss_sector", "RINGSS sector")]
    return refs


def add_derived_columns(df, L0_options=DEFAULT_OUTER_SCALES, refs=None):
    out = df.copy()
    out["psf_fwhm"] = out["psf_sigma_median"].astype(float) * SIGMA_TO_FWHM * PLATESCALE
    out["fwhm_zenith_500nm"] = fwhm_zenith_500nm(
        out["psf_sigma_median"], out["airmass"], out["band"]
    )
    out["dimm_r0"] = fried_parameter(out["dimm_seeing"])
    for L0 in L0_options:
        out[outer_scale_tag(L0)] = out["dimm_seeing"].to_numpy(
            dtype=float
        ) * outer_scale_correction(out["dimm_seeing"], L0)
    for tag, _ in refs or make_references(L0_options):
        if tag not in out:
            out[tag] = np.nan
        out[f"delta_{tag}"] = out["fwhm_zenith_500nm"] - out[tag]
        q2 = out["fwhm_zenith_500nm"] ** 2 - out[tag] ** 2
        out[f"excess_quad_{tag}"] = np.sqrt(q2.where(q2 > 0))
    out["time"] = pd.to_datetime(
        out["exp_midpt"], utc=True, errors="coerce"
    ).dt.tz_localize(None)
    out["rel_wind_az"] = (
        (out["azimuth"] - out["wind_dir"] + 180.0) % 360.0
    ) - 180.0  # 0 = into the wind
    return out


@dataclass
class QualityCuts:
    dimm_range: tuple = (0.3, 4.0)
    ringss_range: tuple = (0.2, 4.0)
    fwhm_range: tuple = (0.4, 4.0)
    airmass_max: float = 2.5
    exclude_bands: tuple = ()
    min_wind_speed: float = (
        None  # the DIMM is unreliable below ~6 m/s; None keeps all winds
    )
    require_dimm: bool = True  # common sample: every row has a valid DIMM ...
    require_ringss: bool = (
        True  # ... and a valid RINGSS, so all references see the same visits
    )


def apply_quality_cuts(df, cuts=QualityCuts(), verbose=True):
    dimm_ok = df["dimm_seeing"].between(*cuts.dimm_range)
    ringss_ok = df["ringss_scint"].between(*cuts.ringss_range)
    steps = [
        ("has PSF FWHM", df["fwhm_zenith_500nm"].notna()),
        ("has wind speed", df["wind_speed"].notna()),
        (
            f"FWHM(zenith,500nm) in {cuts.fwhm_range}",
            df["fwhm_zenith_500nm"].between(*cuts.fwhm_range),
        ),
        (f"airmass <= {cuts.airmass_max}", df["airmass"] <= cuts.airmass_max),
    ]
    if cuts.require_dimm:
        steps.append((f"DIMM present and in {cuts.dimm_range}", dimm_ok))
    if cuts.require_ringss:
        steps.append((f"RINGSS present and in {cuts.ringss_range}", ringss_ok))
    if cuts.exclude_bands:
        steps.append(
            (f"band not in {cuts.exclude_bands}", ~df["band"].isin(cuts.exclude_bands))
        )
    if cuts.min_wind_speed is not None:
        steps.append(
            (
                f"wind_speed >= {cuts.min_wind_speed}",
                df["wind_speed"] >= cuts.min_wind_speed,
            )
        )
    keep = pd.Series(True, index=df.index)
    rows = []
    for name, mask in steps:
        mask = mask.fillna(False)
        rows.append((name, int((keep & ~mask).sum()), int((keep & mask).sum())))
        keep &= mask
    out = df[keep].copy()
    # Out-of-range DIMM / RINGSS values are blanked (not dropped) when not required.
    out.loc[
        ~dimm_ok[keep],
        [c for c in out.columns if c.startswith("dimm_") and c != "dimm_r0"],
    ] = np.nan
    out.loc[~ringss_ok[keep], ["ringss_scint", "ringss_sector", "ringss_free"]] = np.nan
    if verbose:
        print(f"Start: {len(df)} visits")
        print(
            pd.DataFrame(rows, columns=["cut", "removed", "remaining"]).to_string(
                index=False
            )
        )
    return out

## Statistics and plotting helpers

In [ ]:
def binned_stats(x, y, bins, min_count=10):
    """Median, 16/84 percentiles and a robust median error (1.2533*1.4826*MAD/sqrt(n)) of y in x bins."""
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    edges = np.asarray(bins, dtype=float)
    idx = np.digitize(x, edges) - 1
    rows = []
    for i in range(len(edges) - 1):
        sel = y[idx == i]
        row = dict(
            lo=edges[i],
            hi=edges[i + 1],
            center=0.5 * (edges[i] + edges[i + 1]),
            n=len(sel),
            median=np.nan,
            p16=np.nan,
            p84=np.nan,
            mean=np.nan,
            median_err=np.nan,
        )
        if len(sel) >= min_count:
            med = np.median(sel)
            mad = np.median(np.abs(sel - med))
            row.update(
                median=med,
                p16=np.percentile(sel, 16),
                p84=np.percentile(sel, 84),
                mean=sel.mean(),
                median_err=1.2533 * 1.4826 * mad / np.sqrt(len(sel)),
            )
        rows.append(row)
    return pd.DataFrame(rows)


def summary_by_reference(df, refs):
    rows = []
    for tag, label in refs:
        d = df[f"delta_{tag}"].dropna()
        rows.append(
            dict(
                reference=label,
                n=len(d),
                median_ref=df[tag].median(),
                median_delta=d.median(),
                p16_delta=d.quantile(0.16),
                p84_delta=d.quantile(0.84),
                mean_delta=d.mean(),
                rms_delta=np.sqrt((d**2).mean()),
                frac_fwhm_below_ref=(d < 0).mean(),
            )
        )
    return pd.DataFrame(rows).set_index("reference")


def _ylabel(prefix):
    return (
        "FWHM(zenith, 500 nm) - reference (arcsec)"
        if prefix == "delta"
        else "sqrt(FWHM^2 - reference^2) (arcsec)"
    )


def plot_delta_vs_wind_hexbin(
    df,
    refs,
    wind_bins=np.arange(0, 20.1, 1.0),
    ylim=(-0.6, 1.2),
    gridsize=40,
    prefix="delta",
    title=None,
    ncols=3,
):
    """One hexbin panel per reference with the binned median (16-84 % band) overlaid."""
    n = len(refs)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(5.2 * ncols, 4.6 * nrows),
        sharex=True,
        sharey=True,
        squeeze=False,
        layout="constrained",
    )
    xmin, xmax = float(wind_bins[0]), float(wind_bins[-1])
    hb = None
    for k, (ax, (tag, label)) in enumerate(zip(axes.flat, refs)):
        y = df[f"{prefix}_{tag}"]
        ok = y.notna() & df["wind_speed"].notna()
        hb = ax.hexbin(
            df.loc[ok, "wind_speed"],
            y[ok],
            gridsize=gridsize,
            extent=(xmin, xmax, *ylim),
            cmap=SEQUENTIAL_BLUE,
            norm=LogNorm(vmin=1),
            mincnt=1,
            linewidths=0.2,
            edgecolors="face",
        )
        st = binned_stats(df.loc[ok, "wind_speed"], y[ok], wind_bins)
        ax.fill_between(
            st["center"], st["p16"], st["p84"], color=SERIES_COLORS[1], alpha=0.18, lw=0
        )
        ax.plot(st["center"], st["median"], color=SURFACE, lw=4, solid_capstyle="round")
        ax.plot(
            st["center"],
            st["median"],
            color=SERIES_COLORS[1],
            lw=2,
            marker="o",
            ms=4,
            label="binned median (16-84 % band)",
        )
        ax.axhline(0, color=INK_MUTED, lw=1, ls="--")
        ax.set_title(f"{label}   (n = {int(ok.sum())})")
        ax.set_xlim(xmin, xmax)
        ax.set_ylim(*ylim)
        if k // ncols == nrows - 1:
            ax.set_xlabel("wind speed (m/s)")
        if k % ncols == 0:
            ax.set_ylabel(_ylabel(prefix))
    for ax in axes.flat[n:]:
        ax.set_visible(False)
    axes[0, 0].legend(loc="upper left")
    cbar = fig.colorbar(
        hb, ax=[a for a in axes.flat if a.get_visible()], pad=0.02, shrink=0.9
    )
    cbar.set_label("visits per hexagon")
    if title:
        fig.suptitle(title)
    return fig


def plot_median_vs_wind(
    df,
    refs,
    wind_bins=np.arange(0, 20.1, 1.0),
    prefix="delta",
    show_band=False,
    ax=None,
    title=None,
):
    """Binned median vs wind speed, one line per reference (error bars: robust SE of the median)."""
    fig, ax = (ax.figure, ax) if ax is not None else plt.subplots(figsize=(8, 5))
    for k, (tag, label) in enumerate(refs):
        st = binned_stats(df["wind_speed"], df[f"{prefix}_{tag}"], wind_bins)
        color = SERIES_COLORS[k % len(SERIES_COLORS)]
        if show_band:
            ax.fill_between(
                st["center"], st["p16"], st["p84"], color=color, alpha=0.12, lw=0
            )
        ax.errorbar(
            st["center"],
            st["median"],
            yerr=st["median_err"],
            color=color,
            marker=SERIES_MARKERS[k % len(SERIES_MARKERS)],
            ms=6,
            capsize=2,
            lw=2,
            label=label,
        )
    ax.axhline(0, color=INK_MUTED, lw=1, ls="--")
    ax.set_xlabel("wind speed (m/s)")
    ax.set_ylabel("median " + _ylabel(prefix))
    ax.set_xlim(float(wind_bins[0]), float(wind_bins[-1]))
    ax.legend(title="reference seeing")
    if title:
        ax.set_title(title)
    return fig


def plot_counts_vs_wind(df, wind_bins=np.arange(0, 20.1, 1.0), ax=None):
    fig, ax = (ax.figure, ax) if ax is not None else plt.subplots(figsize=(8, 2.6))
    ax.hist(
        df["wind_speed"].dropna(), bins=wind_bins, color=SERIES_COLORS[0], rwidth=0.9
    )
    ax.set_xlabel("wind speed (m/s)")
    ax.set_ylabel("visits")
    ax.set_xlim(float(wind_bins[0]), float(wind_bins[-1]))
    return fig


def plot_delta_histograms(df, refs, bins=np.arange(-1.0, 1.5, 0.02), prefix="delta"):
    fig, ax = plt.subplots(figsize=(8, 5))
    for k, (tag, label) in enumerate(refs):
        d = df[f"{prefix}_{tag}"].dropna()
        color = SERIES_COLORS[k % len(SERIES_COLORS)]
        ax.hist(
            d,
            bins=bins,
            histtype="step",
            lw=2,
            color=color,
            label=f'{label}: median {d.median():+.3f}"',
        )
        ax.axvline(d.median(), color=color, lw=1, ls=":")
    ax.axvline(0, color=INK_MUTED, lw=1, ls="--")
    ax.set_xlabel(_ylabel(prefix))
    ax.set_ylabel("visits")
    ax.legend()
    return fig


def plot_xy_hexbin(df, pairs, lim=(0.3, 2.5), gridsize=45, ncols=3):
    """Hexbin of y vs x for each (xcol, ycol, xlabel, ylabel, title) with the 1:1 line and binned median."""
    n = len(pairs)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(4.8 * ncols, 4.6 * nrows),
        squeeze=False,
        layout="constrained",
    )
    xbins = np.linspace(lim[0], lim[1], 23)
    hb = None
    for ax, (xcol, ycol, xlabel, ylabel, title) in zip(axes.flat, pairs):
        ok = df[xcol].notna() & df[ycol].notna()
        hb = ax.hexbin(
            df.loc[ok, xcol],
            df.loc[ok, ycol],
            gridsize=gridsize,
            extent=(*lim, *lim),
            cmap=SEQUENTIAL_BLUE,
            norm=LogNorm(vmin=1),
            mincnt=1,
            linewidths=0.2,
            edgecolors="face",
        )
        st = binned_stats(df.loc[ok, xcol], df.loc[ok, ycol], xbins)
        ax.plot(lim, lim, color=INK_MUTED, lw=1, ls="--", label="1:1")
        ax.plot(st["center"], st["median"], color=SURFACE, lw=4)
        ax.plot(
            st["center"],
            st["median"],
            color=SERIES_COLORS[1],
            lw=2,
            marker="o",
            ms=4,
            label="binned median",
        )
        ax.set_title(f"{title}   (n = {int(ok.sum())})")
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_xlim(*lim)
        ax.set_ylim(*lim)
        ax.set_aspect("equal")
    for ax in axes.flat[n:]:
        ax.set_visible(False)
    axes[0, 0].legend(loc="upper left")
    cbar = fig.colorbar(
        hb, ax=[a for a in axes.flat if a.get_visible()], pad=0.02, shrink=0.9
    )
    cbar.set_label("visits per hexagon")
    return fig


def plot_median_vs_wind_by_band(
    df, tag, label, wind_bins=np.arange(0, 20.1, 2.0), prefix="delta", min_count=20
):
    fig, ax = plt.subplots(figsize=(8, 5))
    for k, band in enumerate(BAND_ORDER):
        sub = df[df["band"] == band]
        if len(sub) < min_count:
            continue
        st = binned_stats(
            sub["wind_speed"], sub[f"{prefix}_{tag}"], wind_bins, min_count=min_count
        )
        ax.errorbar(
            st["center"],
            st["median"],
            yerr=st["median_err"],
            color=SERIES_COLORS[k],
            marker=SERIES_MARKERS[k],
            ms=6,
            capsize=2,
            lw=2,
            label=f"{band} (n = {len(sub)})",
        )
    ax.axhline(0, color=INK_MUTED, lw=1, ls="--")
    ax.set_xlabel("wind speed (m/s)")
    ax.set_ylabel(f"median FWHM - {label} (arcsec)")
    ax.set_xlim(float(wind_bins[0]), float(wind_bins[-1]))
    ax.legend(title="band", ncol=2)
    return fig


def nightly_summary(df, refs):
    agg = {"wind_speed": "median", "fwhm_zenith_500nm": "median", "visit_id": "size"}
    for tag, _ in refs:
        agg[tag] = "median"
        agg[f"delta_{tag}"] = "median"
    out = df.groupby("day_obs").agg(agg).rename(columns={"visit_id": "n_visits"})
    out["date"] = pd.to_datetime(out.index.astype(str))
    return out


def _break_gaps(nightly, max_gap_days=3):
    """Insert NaN rows after gaps longer than max_gap_days so lines break instead of bridging."""
    nightly = nightly.sort_values("date")
    gaps = nightly["date"].diff().dt.days > max_gap_days
    if not gaps.any():
        return nightly
    pieces = []
    for i, (_, row) in enumerate(nightly.iterrows()):
        if gaps.iloc[i]:
            blank = row.copy()
            blank[:] = np.nan
            blank["date"] = row["date"] - pd.Timedelta(days=1)
            pieces.append(blank)
        pieces.append(row)
    return pd.DataFrame(pieces).reset_index(drop=True)


def plot_nightly(df, refs, top_refs=("dimm_none", "ringss_scint")):
    """Night-by-night medians: FWHM and raw references on top, FWHM - reference per option below, wind at the bottom."""
    nightly = _break_gaps(nightly_summary(df, refs))
    labels = dict(refs)
    fig, axes = plt.subplots(
        3, 1, figsize=(12, 9), sharex=True, gridspec_kw={"height_ratios": [2, 2, 1.2]}
    )
    ax = axes[0]
    ax.plot(
        nightly["date"],
        nightly["fwhm_zenith_500nm"],
        marker="o",
        ms=4,
        lw=1.2,
        color=INK,
        label="FWHM(zenith, 500 nm)",
    )
    for k, tag in enumerate([t for t in top_refs if t in labels]):
        ax.plot(
            nightly["date"],
            nightly[tag],
            marker=SERIES_MARKERS[k + 1],
            ms=4,
            lw=1.2,
            color=SERIES_COLORS[[t for t, _ in refs].index(tag)],
            label=labels[tag],
        )
    ax.set_ylabel("nightly median (arcsec)")
    ax.legend(ncol=3)
    ax = axes[1]
    for k, (tag, label) in enumerate(refs):
        ax.plot(
            nightly["date"],
            nightly[f"delta_{tag}"],
            marker=SERIES_MARKERS[k],
            ms=4,
            lw=1.2,
            color=SERIES_COLORS[k],
            label=label,
        )
    ax.axhline(0, color=INK_MUTED, lw=1, ls="--")
    ax.set_ylabel("median FWHM - reference (arcsec)")
    ax.legend(ncol=3, fontsize=9)
    ax = axes[2]
    ax.plot(
        nightly["date"],
        nightly["wind_speed"],
        marker="o",
        ms=4,
        lw=1.2,
        color=SERIES_COLORS[2],
    )
    ax.set_ylabel("median wind (m/s)")
    ax.set_xlabel("night (day_obs)")
    fig.align_ylabels(axes)
    return fig


def plot_median_vs_rel_wind_az(
    df, tag, label, az_bins=np.arange(-180, 181, 30), min_wind=6.0, prefix="delta"
):
    """FWHM - reference against telescope azimuth relative to the wind (0 = into the wind)."""
    sub = df[df["wind_speed"] >= min_wind]
    st = binned_stats(sub["rel_wind_az"], sub[f"{prefix}_{tag}"], az_bins)
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.fill_between(
        st["center"], st["p16"], st["p84"], color=SERIES_COLORS[0], alpha=0.12, lw=0
    )
    ax.errorbar(
        st["center"],
        st["median"],
        yerr=st["median_err"],
        color=SERIES_COLORS[0],
        marker="o",
        ms=6,
        capsize=2,
        lw=2,
        label=f"median (wind >= {min_wind:g} m/s, n = {len(sub)})",
    )
    ax.axhline(0, color=INK_MUTED, lw=1, ls="--")
    ax.set_xlabel("telescope azimuth - wind direction (deg; 0 = into the wind)")
    ax.set_ylabel(f"FWHM - {label} (arcsec)")
    ax.set_xlim(-180, 180)
    ax.set_xticks(np.arange(-180, 181, 60))
    ax.legend()
    return fig

## Parameters and data

In [ ]:
day_obs_min = parse_day_obs(day_obs_min)
day_obs_max = parse_day_obs(day_obs_max)
programs = parse_programs(science_programs)
L0_options = parse_outer_scales(outer_scales)
ringss_window_minutes = float(ringss_window_minutes)
ringss_at_zenith = str(ringss_at_zenith).strip().lower() in ("true", "1", "yes")

DIMM_REFS = make_references(L0_options, include_ringss=False)
RINGSS_REFS = [
    ("ringss_scint", "RINGSS scint"),
    ("ringss_sector", "RINGSS sector"),
    ("ringss_free", "RINGSS free atm"),
]  # fwhmScintillation, fwhmSector
REFS = DIMM_REFS + RINGSS_REFS  # every reference, in plotting order
MAIN_REFS = (
    DIMM_REFS + RINGSS_REFS[:1]
)  # the headline comparison: DIMM options + RINGSS total

cache_tag = (
    f"{day_obs_min}_{day_obs_max}_{'_'.join(p.replace('BLOCK-', '') for p in programs)}"
)
cache_dir = (
    os.path.join(NOTEBOOK_DIR, "cache") if (use_cache and not_times_square) else None
)

print(f"Nights {day_obs_min}-{day_obs_max}; programs {programs}")
print("References:", [label for _, label in REFS])

In [ ]:
if data_source == "consdb":
    consdb = get_consdb_client()
    raw = cached(
        cache_dir and os.path.join(cache_dir, f"visits_{cache_tag}.parquet"),
        lambda: fetch_visits(consdb, day_obs_min, day_obs_max, programs),
        use_cache,
    )
    efd = get_efd_client(efd_name)
    ringss = await cached_async(
        cache_dir
        and os.path.join(cache_dir, f"ringss_{day_obs_min}_{day_obs_max}.parquet"),
        lambda: fetch_ringss(efd, day_obs_min, day_obs_max),
        use_cache,
    )
    visits = match_ringss_to_visits(
        raw, ringss, ringss_window_minutes, ringss_at_zenith
    )
else:
    raise ValueError(f"unknown data_source {data_source!r}")

print(f"{len(visits)} visits on {visits['day_obs'].nunique()} nights")
visits.head()

In [ ]:
# What came back, per program and band, and how complete the key columns are.
print(pd.crosstab(visits["science_program"], visits["band"], margins=True))
print()
print("fraction of visits with a value:")
print(
    visits[
        [
            "psf_sigma_median",
            "dimm_seeing",
            "ringss_scint",
            "ringss_sector",
            "wind_speed",
            "wind_dir",
            "seeing_zenith_500nm_median",
            "guider_total_seeing",
        ]
    ]
    .notna()
    .mean()
    .round(3)
)
if ringss is not None and len(ringss):
    print(
        f"\nRINGSS: {len(ringss)} measurements, {ringss['time'].min()} to {ringss['time'].max()}; "
        f"median cadence {ringss['time'].diff().median()}; visits with >= 1 match: {(visits['ringss_n'] > 0).mean():.2f}"
    )
    print(
        ringss[
            ["fwhmScintillation", "fwhmSector", "fwhmFree", "wind", "zenithDistance"]
        ]
        .describe()
        .round(3)
    )

## Derived quantities and quality cuts

The cuts drop visits without a PSF or wind measurement, clip implausible seeing values, and by default keep
only the **common sample** where both the DIMM and RINGSS are valid, so every reference is judged on the
same visits. Set `require_dimm`/`require_ringss` to `False` to use everything each monitor has (out-of-range
values are then blanked rather than dropped). `min_wind_speed` is `None` so the low-wind regime, where the
DIMM is known to be unreliable (below roughly 6 m/s), stays visible on the wind axis.

In [ ]:
visits = add_derived_columns(visits, L0_options, REFS)

cuts = QualityCuts(
    dimm_range=(0.3, 4.0),
    ringss_range=(0.2, 4.0),
    fwhm_range=(0.4, 4.0),
    airmass_max=2.5,
    exclude_bands=(),  # e.g. ("u",)
    min_wind_speed=None,  # e.g. 6.0 to keep only winds where the DIMM is trusted
    require_dimm=True,
    require_ringss=True,
)
sel = apply_quality_cuts(visits, cuts)

In [ ]:
# Sanity check on the outer-scale correction: factor applied to the DIMM at representative seeing values.
probe = pd.DataFrame({"dimm_seeing": [0.5, 0.7, 1.0, 1.5, 2.0]})
probe["r0 (m)"] = fried_parameter(probe["dimm_seeing"])
for L0 in L0_options:
    probe[outer_scale_label(L0)] = outer_scale_correction(probe["dimm_seeing"], L0)
probe.round(3)

In [ ]:
summary_by_reference(sel, REFS).round(3)

## Δ = FWHM(zenith, 500 nm) − reference versus wind speed

Hexbin density of all visits (log colour scale) with the binned median and the 16–84 % band overlaid,
one panel per reference: the DIMM outer-scale options in the top row, RINGSS below.

In [ ]:
wind_bins = np.arange(0, 20.1, 1.0)

fig = plot_delta_vs_wind_hexbin(
    sel,
    REFS,
    wind_bins=wind_bins,
    ylim=(-0.6, 1.2),
    ncols=len(L0_options),
    title=f"{day_obs_min}-{day_obs_max}, {', '.join(programs)}",
)
plt.show()

In [ ]:
# Medians on one axis (error bars: robust standard error of the median), visit counts per wind bin beneath.
fig, (ax0, ax1) = plt.subplots(
    2, 1, figsize=(8.5, 7.5), sharex=True, gridspec_kw={"height_ratios": [3, 1]}
)
plot_median_vs_wind(
    sel,
    REFS,
    wind_bins=wind_bins,
    ax=ax0,
    title="median FWHM(zenith, 500 nm) - reference vs wind speed",
)
plot_counts_vs_wind(sel, wind_bins=wind_bins, ax=ax1)
ax0.set_xlabel("")
plt.show()

In [ ]:
# Binned-median table for the record: one row per wind bin, one column per reference.
tables = []
for tag, label in REFS:
    t = binned_stats(sel["wind_speed"], sel[f"delta_{tag}"], wind_bins)
    t.insert(0, "reference", label)
    tables.append(t)
binned = pd.concat(tables, ignore_index=True)
(
    binned.pivot(index="center", columns="reference", values="median")[
        [label for _, label in REFS]
    ]
    .round(3)
    .join(binned[binned["reference"] == REFS[0][1]].set_index("center")["n"])
)

## Distribution of Δ and FWHM against each reference

The histograms show how each reference shifts the whole distribution; the FWHM-vs-reference panels show
whether the relation is a constant offset or a slope (the 1:1 line is the perfect-prediction case).

In [ ]:
fig = plot_delta_histograms(sel, REFS, bins=np.arange(-1.0, 1.5, 0.02))
plt.show()

In [ ]:
pairs = [
    (
        tag,
        "fwhm_zenith_500nm",
        f"{label} (arcsec)",
        "FWHM(zenith, 500 nm) (arcsec)",
        label,
    )
    for tag, label in REFS
]
fig = plot_xy_hexbin(sel, pairs, lim=(0.3, 2.5), ncols=len(L0_options))
plt.show()

## DIMM versus RINGSS directly

Which outer-scale option brings the DIMM closest to RINGSS, and how the two monitors' disagreement depends on
wind. Both are measured at zenith / 500 nm so the comparison is independent of the telescope.

In [ ]:
for tag, _ in DIMM_REFS:
    sel[f"dimm_minus_ringss_{tag}"] = sel[tag] - sel["ringss_scint"]

pairs = [
    ("ringss_scint", tag, "RINGSS total (arcsec)", f"{label} (arcsec)", label)
    for tag, label in DIMM_REFS
]
fig = plot_xy_hexbin(sel, pairs, lim=(0.3, 2.5), ncols=len(L0_options))
plt.show()

fig, ax = plt.subplots(figsize=(8.5, 5))
for k, (tag, label) in enumerate(DIMM_REFS):
    st = binned_stats(sel["wind_speed"], sel[f"dimm_minus_ringss_{tag}"], wind_bins)
    ax.errorbar(
        st["center"],
        st["median"],
        yerr=st["median_err"],
        color=SERIES_COLORS[k],
        marker=SERIES_MARKERS[k],
        ms=6,
        capsize=2,
        lw=2,
        label=label,
    )
ax.axhline(0, color=INK_MUTED, lw=1, ls="--")
ax.set_xlabel("wind speed (m/s)")
ax.set_ylabel("median DIMM - RINGSS total (arcsec)")
ax.set_xlim(float(wind_bins[0]), float(wind_bins[-1]))
ax.legend(title="DIMM option")
plt.show()

pd.DataFrame(
    {
        label: sel[f"dimm_minus_ringss_{tag}"].describe(percentiles=[0.16, 0.5, 0.84])
        for tag, label in DIMM_REFS
    }
).round(3)

## Quadrature excess

The same wind dependence for $\sqrt{\mathrm{FWHM}^2 - \mathrm{ref}^2}$, the extra blur that would have to be
added in quadrature to the reference to reproduce the delivered PSF. Visits where the reference exceeds the
delivered FWHM are undefined here and are dropped from these panels.

In [ ]:
fig = plot_delta_vs_wind_hexbin(
    sel, MAIN_REFS, wind_bins=wind_bins, ylim=(0.0, 1.4), prefix="excess_quad", ncols=2
)
plt.show()

fig = plot_median_vs_wind(
    sel,
    MAIN_REFS,
    wind_bins=wind_bins,
    prefix="excess_quad",
    title="median quadrature excess vs wind speed",
)
plt.show()

## Breakdown by band and by night

The per-band plot uses one reference at a time (change `BAND_REF`). A band-dependent offset would point at the
$\lambda^{0.2}$ scaling or at band-specific optics terms rather than the atmosphere.

In [ ]:
BAND_REF = DIMM_REFS[-1]  # e.g. RINGSS_REFS[0] for RINGSS
fig = plot_median_vs_wind_by_band(sel, *BAND_REF, wind_bins=np.arange(0, 20.1, 2.0))
plt.show()

In [ ]:
fig = plot_nightly(sel, MAIN_REFS)
plt.show()

nightly_summary(sel, MAIN_REFS).drop(columns="date").round(3)